In [3]:
import psycopg2
import random
from datetime import datetime, timedelta
import uuid
from faker import Faker

fake = Faker()

# Подключение к БД
conn = psycopg2.connect(
    dbname="forum_logs",
    user="user",
    password="password",
    host="localhost"
)
cursor = conn.cursor()

# Параметры генерации
DAYS = 30
MIN_ACTIONS_PER_TYPE = 5
TOPIC_CREATE_ERRORS = 2

def generate_data():
    start_date = datetime.now() - timedelta(days=DAYS)
    
    for day in range(DAYS):
        current_date = start_date + timedelta(days=day)
        print(f"Generating data for {current_date.date()}")
        
        # Генерация пользователей
        users = []
        for _ in range(random.randint(MIN_ACTIONS_PER_TYPE, MIN_ACTIONS_PER_TYPE * 2)):
            username = fake.user_name()
            email = fake.email()
            cursor.execute(
                "INSERT INTO users (username, email, password_hash, registration_date) "
                "VALUES (%s, %s, %s, %s) RETURNING user_id",
                (username, email, fake.sha256(), current_date)
            )
            user_id = cursor.fetchone()[0]
            users.append(user_id)
            
            # Лог регистрации
            cursor.execute(
                "INSERT INTO user_logs (action_type_id, user_id, action_time, server_response, ip_address, user_agent) "
                "VALUES (2, %s, %s, 'success', %s, %s)",
                (user_id, current_date, fake.ipv4(), fake.user_agent())
            )
        
        # Генерация анонимных пользователей
        anon_users = []
        for _ in range(random.randint(MIN_ACTIONS_PER_TYPE * 3, MIN_ACTIONS_PER_TYPE * 5)):
            cursor.execute(
                "INSERT INTO anonymous_users (session_id, ip_address, user_agent, first_seen) "
                "VALUES (%s, %s, %s, %s) RETURNING anon_id",
                (str(uuid.uuid4()), fake.ipv4(), fake.user_agent(), current_date)
            )
            anon_id = cursor.fetchone()[0]
            anon_users.append(anon_id)
            
            # Лог первого визита
            cursor.execute(
                "INSERT INTO user_logs (action_type_id, anon_id, action_time, server_response, ip_address, user_agent) "
                "VALUES (1, %s, %s, 'success', %s, %s)",
                (anon_id, current_date, fake.ipv4(), fake.user_agent())
            )
        
        # Генерация тем
        topics = []
        logged_in_users = random.sample(users, min(len(users), random.randint(3, 10)))
        
        # Успешное создание тем
        for user_id in logged_in_users:
            for _ in range(random.randint(1, 3)):
                cursor.execute(
                    "INSERT INTO topics (title, content, created_by, created_at) "
                    "VALUES (%s, %s, %s, %s) RETURNING topic_id",
                    (fake.sentence(), fake.text(), user_id, current_date)
                )
                topic_id = cursor.fetchone()[0]
                topics.append(topic_id)
                
                # Лог создания темы
                cursor.execute(
                    "INSERT INTO user_logs (action_type_id, user_id, action_time, entity_type, entity_id, server_response, ip_address, user_agent) "
                    "VALUES (5, %s, %s, 'topic', %s, 'success', %s, %s)",
                    (user_id, current_date, topic_id, fake.ipv4(), fake.user_agent())
                )
        
        # Ошибки создания тем (незалогиненные)
        for _ in range(TOPIC_CREATE_ERRORS):
            anon_id = random.choice(anon_users)
            cursor.execute(
                "INSERT INTO user_logs (action_type_id, anon_id, action_time, server_response, additional_info, ip_address, user_agent) "
                "VALUES (5, %s, %s, 'error', 'User not logged in', %s, %s)",
                (anon_id, current_date, fake.ipv4(), fake.user_agent())
            )
        
        # Генерация сообщений
        for _ in range(random.randint(MIN_ACTIONS_PER_TYPE * 5, MIN_ACTIONS_PER_TYPE * 10)):
            topic_id = random.choice(topics)
            is_anon = random.choice([True, False])
            
            if is_anon:
                anon_id = random.choice(anon_users)
                cursor.execute(
                    "INSERT INTO messages (topic_id, content, created_at, anon_id) "
                    "VALUES (%s, %s, %s, %s) RETURNING message_id",
                    (topic_id, fake.text(), current_date, anon_id)
                )
                message_id = cursor.fetchone()[0]
                
                cursor.execute(
                    "INSERT INTO user_logs (action_type_id, anon_id, action_time, entity_type, entity_id, server_response, ip_address, user_agent) "
                    "VALUES (8, %s, %s, 'message', %s, 'success', %s, %s)",
                    (anon_id, current_date, message_id, fake.ipv4(), fake.user_agent())
                )
            else:
                user_id = random.choice(users)
                cursor.execute(
                    "INSERT INTO messages (topic_id, content, created_at, user_id) "
                    "VALUES (%s, %s, %s, %s) RETURNING message_id",
                    (topic_id, fake.text(), current_date, user_id)
                )
                message_id = cursor.fetchone()[0]
                
                cursor.execute(
                    "INSERT INTO user_logs (action_type_id, user_id, action_time, entity_type, entity_id, server_response, ip_address, user_agent) "
                    "VALUES (8, %s, %s, 'message', %s, 'success', %s, %s)",
                    (user_id, current_date, message_id, fake.ipv4(), fake.user_agent())
                )
        
        # Другие действия
        for user_id in random.sample(users, min(len(users), random.randint(MIN_ACTIONS_PER_TYPE, MIN_ACTIONS_PER_TYPE * 3))):
            # Логины/логауты
            for action_type in [3, 4]:  # login, logout
                for _ in range(random.randint(1, 3)):
                    cursor.execute(
                        "INSERT INTO user_logs (action_type_id, user_id, action_time, server_response, ip_address, user_agent) "
                        "VALUES (%s, %s, %s, 'success', %s, %s)",
                        (action_type, user_id, current_date, fake.ipv4(), fake.user_agent())
                    )
        
        # Просмотры тем
        for _ in range(random.randint(MIN_ACTIONS_PER_TYPE * 5, MIN_ACTIONS_PER_TYPE * 10)):
            topic_id = random.choice(topics)
            is_anon = random.choice([True, False])
            
            if is_anon:
                anon_id = random.choice(anon_users)
                cursor.execute(
                    "INSERT INTO user_logs (action_type_id, anon_id, action_time, entity_type, entity_id, server_response, ip_address, user_agent) "
                    "VALUES (6, %s, %s, 'topic', %s, 'success', %s, %s)",
                    (anon_id, current_date, topic_id, fake.ipv4(), fake.user_agent())
                )
            else:
                user_id = random.choice(users)
                cursor.execute(
                    "INSERT INTO user_logs (action_type_id, user_id, action_time, entity_type, entity_id, server_response, ip_address, user_agent) "
                    "VALUES (6, %s, %s, 'topic', %s, 'success', %s, %s)",
                    (user_id, current_date, topic_id, fake.ipv4(), fake.user_agent())
                )
        
        conn.commit()

if __name__ == "__main__":
    generate_data()
    cursor.close()
    conn.close()

OperationalError: connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?


In [ ]:
import psycopg2
import pandas as pd
from datetime import datetime

def aggregate(start_date, end_date):
    conn = psycopg2.connect(
        dbname='forum_logs',
        user='user',
        password='password',
        host='localhost',
        port='5432'
    )
    query = f"""
    SELECT
      DATE(created_at) as day,
      COUNT(*) FILTER (WHERE action_type = 'register') as registrations,
      COUNT(*) FILTER (WHERE action_type = 'post_message') as total_messages,
      COUNT(*) FILTER (WHERE action_type = 'post_message' AND user_id IS NULL) as anon_messages,
      COUNT(*) FILTER (WHERE action_type = 'create_topic' AND result = 'success') as new_topics
    FROM logs
    WHERE created_at::date BETWEEN '{start_date}' AND '{end_date}'
    GROUP BY day
    ORDER BY day
    """
    df = pd.read_sql_query(query, conn)
    df['anon_percentage'] = (df['anon_messages'] / df['total_messages'] * 100).round(2)
    df['topics_change_%'] = df['new_topics'].pct_change().fillna(0).round(2) * 100
    df.to_csv('aggregated_logs.csv', index=False)
    conn.close()

# Пример вызова:
# aggregate('2024-03-01', '2024-03-31')
